<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/02_machine_learning/reinforcement_learning/experiment_reinforce_policy_gradient_pytorch_basic_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 32),
            nn.ReLU(),
            nn.Linear(32, action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        return self.net(x)


class REINFORCE:
    def __init__(self, state_dim, action_dim):
        self.policy = PolicyNet(state_dim, action_dim)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=0.01)
        self.gamma = 0.99

    def train(self, episodes=100):
        for _ in range(episodes):
            log_probs = []
            rewards = []

            state = np.random.rand(1)

            for _ in range(10):
                state_tensor = torch.tensor(state, dtype=torch.float32)

                probs = self.policy(state_tensor)
                dist = torch.distributions.Categorical(probs)

                action = dist.sample()
                log_prob = dist.log_prob(action)

                reward = 1 if action.item() == 1 else 0

                log_probs.append(log_prob)
                rewards.append(reward)

                state = np.random.rand(1)

            # Compute returns
            returns = []
            G = 0
            for r in reversed(rewards):
                G = r + self.gamma * G
                returns.insert(0, G)

            returns = torch.tensor(returns)

            loss = 0
            for log_prob, G in zip(log_probs, returns):
                loss -= log_prob * G

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()


if __name__ == "__main__":
    agent = REINFORCE(state_dim=1, action_dim=2)
    agent.train()